# 2. Exploração e avaliação da qualidade dos dados

Nesta segunda fase do projeto é aprofundada a análise do conjunto de dados, com foco na avaliação da sua qualidade, consistência e adequação às etapas seguintes.

Serão analisados aspetos relacionados com valores omissos, registos duplicados, tipos de dados, categorias, estatística descritiva e possíveis valores extremos.

Será também avaliada a necessidade de transformação ou normalização de determinadas variáveis, tendo em conta as características do dataset e as técnicas que serão utilizadas posteriormente.

Esta fase integra as etapas de **Data Understanding** e **Data Preparation** da abordagem CRISP-DM e tem como objetivo apoiar decisões de preparação dos dados de forma estruturada e justificada.

In [1]:
# 1. imports

import numpy as np
import pandas as pd
import os
RANDOM_STATE = 42

In [2]:
# 2. localização do ficheiro

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/martimf/dataset-c-obesity-csv/Dataset_C_Obesity.csv


In [3]:
# 3. leitura do dataset

DATA_PATH = "/kaggle/input/datasets/martimf/dataset-c-obesity-csv/Dataset_C_Obesity.csv"

df = pd.read_csv(DATA_PATH)

In [4]:
# 4. primeiros 5 registos

df.head()

,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,CH2O,FAF,MTRANS,NObeyesdad
0,Male,23.826684,1.783609,120.921535,yes,yes,2.591292,2.956422,Sometimes,2.310830,1.485240,Public_Transportation,Obesity_Type_II
1,Female,26.591628,1.700000,78.008388,yes,yes,2.734314,3.000000,Sometimes,3.000000,0.897702,Public_Transportation,Overweight_Level_II
2,Male,31.264628,1.803129,91.052215,yes,yes,2.000000,3.000000,Sometimes,1.199517,0.047738,Public_Transportation,Overweight_Level_II
3,Female,18.634286,1.669354,126.088301,yes,yes,3.000000,3.000000,Sometimes,1.144539,0.922014,Public_Transportation,Obesity_Type_III
4,Male,17.580627,1.770324,55.695253,yes,yes,2.000000,4.000000,Sometimes,2.369627,2.000000,Automobile,Insufficient_Weight


## 2.1 Análise dos valores omissos

Na exploração inicial foram identificados valores omissos nas variáveis `Age`, 
`CAEC` e `CH2O`. Cada uma apresenta 15 valores em falta, correspondendo a 
aproximadamente 0,71% dos registos.

Nesta etapa pretende-se analisar com maior detalhe a distribuição desses valores 
omissos, antes de definir a estratégia mais adequada para o seu tratamento.

In [5]:
# 3. registos com valores omissos

registos_omissos = df[df.isna().any(axis=1)]

print(f"Número de registos com pelo menos um valor omisso: {len(registos_omissos)}")

registos_omissos

Número de registos com pelo menos um valor omisso: 45


,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,CH2O,FAF,MTRANS,NObeyesdad
6,Female,NaN,1.722461,51.881263,yes,yes,2.663421,3.000000,Frequently,1.041110,0.794402,Public_Transportation,Insufficient_Weight
72,Male,NaN,1.820930,108.742005,yes,yes,2.000000,1.255350,Sometimes,2.497065,1.000000,Public_Transportation,Obesity_Type_I
73,Female,NaN,1.644560,88.119139,yes,yes,2.693859,1.000283,Sometimes,3.000000,1.201403,Public_Transportation,Obesity_Type_I
101,Female,30.711381,1.536819,65.912688,yes,no,2.030256,2.948721,NaN,1.984323,0.986414,Public_Transportation,Overweight_Level_II
121,Male,21.473078,1.694423,80.311273,yes,yes,2.000000,2.646717,Sometimes,NaN,2.627515,Public_Transportation,Overweight_Level_II
128,Male,18.000000,1.781543,50.869704,no,yes,2.052932,3.000000,NaN,2.000000,0.520408,Public_Transportation,Insufficient_Weight
165,Male,20.000000,1.770000,70.000000,yes,yes,1.000000,1.000000,NaN,2.000000,1.000000,Public_Transportation,Normal_Weight
216,Male,38.712160,1.770278,117.291880,yes,yes,2.252382,2.842848,Sometimes,NaN,0.919388,Automobile,Obesity_Type_II
451,Female,22.989846,1.650000,80.000000,yes,yes,2.000000,3.000000,Sometimes,NaN,0.146919,Public_Transportation,Overweight_Level_II
516,Female,NaN,1.583943,81.936398,yes,yes,2.490507,2.974204,Sometimes,1.846754,0.000000,Automobile,Obesity_Type_I


In [6]:
# 4. número de valores omissos por registo

omissos_por_registo = (
    registos_omissos
    .isna()
    .sum(axis=1)
    .value_counts()
    .sort_index()
)

omissos_por_registo

1    45
Name: count, dtype: int64

### Interpretação

Foram identificados **45 registos com valores omissos**. A análise da sua 
distribuição mostrou que cada um destes registos apresenta apenas **um valor 
em falta**.

Os valores omissos encontram-se distribuídos por três variáveis: `Age`, 
`CAEC` e `CH2O`, com 15 ocorrências em cada variável. Assim, não existem 
registos com ausência simultânea de informação em várias destas variáveis.

Dado que os valores omissos representam uma proporção reduzida do conjunto 
de dados, será analisada uma estratégia de tratamento que permita preservar 
o maior número possível de registos.

In [12]:
df[["Age", "CH2O"]].describe()

,Age,CH2O
count,2096.000000,2096.000000
mean,24.325033,2.008331
std,6.344150,0.612909
min,14.000000,1.000000
25%,19.960667,1.587615
50%,22.815036,2.000000
75%,26.000000,2.476711
max,61.000000,3.000000


In [13]:
# 6. distribuição da variável CAEC

df["CAEC"].value_counts(dropna=False)

CAEC
Sometimes     1753
Frequently     239
Always          53
no              51
NaN             15
Name: count, dtype: int64

### Estratégia de tratamento dos valores omissos

Para as variáveis numéricas `Age` e `CH2O`, optou-se pela imputação através 
da **mediana**. Em `Age`, a mediana é menos sensível à influência das idades 
mais elevadas observadas no dataset. Em `CH2O`, embora a média e a mediana 
sejam muito próximas, foi igualmente utilizada a mediana, mantendo uma 
estratégia consistente e robusta.

Para a variável categórica `CAEC`, optou-se pela **moda**, uma vez que 
`Sometimes` é claramente a categoria mais frequente.

Desta forma, é possível tratar os valores omissos sem eliminar os 45 registos 
afetados e, consequentemente, sem perder a restante informação disponível 
nessas observações.

In [14]:
# 7. tratamento dos valores omissos

mediana_age = df["Age"].median()
mediana_ch2o = df["CH2O"].median()
moda_caec = df["CAEC"].mode()[0]

df["Age"] = df["Age"].fillna(mediana_age)
df["CH2O"] = df["CH2O"].fillna(mediana_ch2o)
df["CAEC"] = df["CAEC"].fillna(moda_caec)

print(f"Mediana de Age: {mediana_age:.2f}")
print(f"Mediana de CH2O: {mediana_ch2o:.2f}")
print(f"Moda de CAEC: {moda_caec}")
print()
# 8. verificação após o tratamento

print("Total de valores omissos:", df.isna().sum().sum())

Mediana de Age: 22.82
Mediana de CH2O: 2.00
Moda de CAEC: Sometimes

Total de valores omissos: 0


### Resultado do tratamento

Após a aplicação das estratégias definidas, os 15 valores omissos de `Age`
foram substituídos pela mediana (22,82), os 15 valores omissos de `CH2O`
pela respetiva mediana (2,00) e os 15 valores omissos de `CAEC` pela moda
(`Sometimes`).

A verificação realizada após a imputação confirmou que o dataset deixou de
apresentar valores omissos. Desta forma, foram preservados os 2111 registos
originais para as etapas seguintes da análise.

## 2.2 Análise dos registos duplicados

Na exploração inicial foram identificados 24 registos duplicados no conjunto
de dados. Antes de proceder à sua eventual remoção, estes registos serão
analisados de forma a avaliar se correspondem a duplicações completas das
observações.

In [15]:
# 9. identificação dos registos duplicados

duplicados = df[df.duplicated(keep=False)]

print(f"Número de linhas envolvidas em duplicações: {len(duplicados)}")

duplicados.sort_values(by=list(df.columns))

Número de linhas envolvidas em duplicações: 33


,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,CH2O,FAF,MTRANS,NObeyesdad
112,Female,16.0,1.66,58.0,no,no,2.0,1.0,Sometimes,1.0,0.0,Walking,Normal_Weight
1218,Female,16.0,1.66,58.0,no,no,2.0,1.0,Sometimes,1.0,0.0,Walking,Normal_Weight
195,Female,18.0,1.62,55.0,yes,yes,2.0,3.0,Frequently,1.0,1.0,Public_Transportation,Normal_Weight
548,Female,18.0,1.62,55.0,yes,yes,2.0,3.0,Frequently,1.0,1.0,Public_Transportation,Normal_Weight
436,Female,21.0,1.52,42.0,no,no,3.0,1.0,Frequently,1.0,0.0,Public_Transportation,Insufficient_Weight
867,Female,21.0,1.52,42.0,no,no,3.0,1.0,Frequently,1.0,0.0,Public_Transportation,Insufficient_Weight
307,Female,21.0,1.52,42.0,no,yes,3.0,1.0,Frequently,1.0,0.0,Public_Transportation,Insufficient_Weight
433,Female,21.0,1.52,42.0,no,yes,3.0,1.0,Frequently,1.0,0.0,Public_Transportation,Insufficient_Weight
1424,Female,21.0,1.52,42.0,no,yes,3.0,1.0,Frequently,1.0,0.0,Public_Transportation,Insufficient_Weight
1787,Female,21.0,1.52,42.0,no,yes,3.0,1.0,Frequently,1.0,0.0,Public_Transportation,Insufficient_Weight


### Estratégia de tratamento dos registos duplicados

A análise identificou 33 linhas envolvidas em situações de duplicação. Este
valor inclui tanto as primeiras ocorrências como as respetivas repetições.

Considerando apenas as ocorrências repetidas, foram identificados 24 registos
duplicados. A inspeção dos dados confirmou que se tratam de duplicações
completas, isto é, os valores coincidem em todos os atributos do dataset.

Assim, optou-se pela remoção das ocorrências duplicadas, mantendo a primeira
ocorrência de cada registo. Esta decisão evita que observações repetidas tenham
um peso artificialmente superior nas análises e modelos posteriores.

In [16]:
# 10. remoção dos registos duplicados

num_registos_antes = len(df)

df = df.drop_duplicates().reset_index(drop=True)

num_registos_depois = len(df)

print(f"Registos antes da remoção: {num_registos_antes}")
print(f"Registos depois da remoção: {num_registos_depois}")
print(f"Registos removidos: {num_registos_antes - num_registos_depois}")
print()
print("Registos duplicados restantes:", df.duplicated().sum())

Registos antes da remoção: 2111
Registos depois da remoção: 2087
Registos removidos: 24

Registos duplicados restantes: 0


In [17]:
# 12. distribuição da variável alvo após remoção dos duplicados

distribuicao_classe = pd.DataFrame({
    "Frequência": df["NObeyesdad"].value_counts(),
    "Percentagem (%)": df["NObeyesdad"].value_counts(normalize=True) * 100
})

distribuicao_classe["Percentagem (%)"] = (
    distribuicao_classe["Percentagem (%)"].round(2)
)

distribuicao_classe

,Frequência,Percentagem (%)
NObeyesdad,,
Obesity_Type_I,351,16.82
Obesity_Type_III,324,15.52
Obesity_Type_II,297,14.23
Overweight_Level_II,290,13.90
Normal_Weight,282,13.51
Overweight_Level_I,276,13.22
Insufficient_Weight,267,12.79


### Resultado do tratamento dos registos duplicados

Após a remoção das ocorrências duplicadas, o número de registos passou de
2111 para 2087, correspondendo à remoção de 24 duplicados.

A distribuição da variável alvo `NObeyesdad` foi novamente analisada após
esta operação. As sete classes continuam a apresentar frequências relativamente
próximas, variando entre 12,79% (`Insufficient_Weight`) e 16,82%
(`Obesity_Type_I`).

Desta forma, a remoção dos registos duplicados não provocou um desequilíbrio
acentuado na distribuição da variável alvo e evita que observações repetidas
tenham um peso artificialmente superior nas análises posteriores.

## 2.3 Análise da consistência dos dados

Após o tratamento dos valores omissos e dos registos duplicados, procede-se
à verificação da consistência dos valores presentes no dataset.

Nesta etapa são analisados os valores das variáveis categóricas e os intervalos
das variáveis numéricas, procurando identificar categorias inesperadas,
valores impossíveis ou outras possíveis inconsistências antes de prosseguir
para as restantes etapas da preparação dos dados.

In [18]:
# 13. verificação das categorias existentes

colunas_categoricas = df.select_dtypes(exclude=np.number).columns

for coluna in colunas_categoricas:
    print(f"\n{coluna}:")
    print(sorted(df[coluna].unique()))

# 14. intervalos das variáveis numéricas

colunas_numericas = df.select_dtypes(include=np.number).columns

intervalos_numericos = pd.DataFrame({
    "Mínimo": df[colunas_numericas].min(),
    "Máximo": df[colunas_numericas].max()
})

intervalos_numericos


Gender:
['Female', 'Male']

family_history_with_overweight:
['no', 'yes']

FAVC:
['no', 'yes']

CAEC:
['Always', 'Frequently', 'Sometimes', 'no']

MTRANS:
['Automobile', 'Bike', 'Motorbike', 'Public_Transportation', 'Walking']

NObeyesdad:
['Insufficient_Weight', 'Normal_Weight', 'Obesity_Type_I', 'Obesity_Type_II', 'Obesity_Type_III', 'Overweight_Level_I', 'Overweight_Level_II']


,Mínimo,Máximo
Age,14.00,61.00
Height,1.45,1.98
Weight,39.00,173.00
FCVC,1.00,3.00
NCP,1.00,4.00
CH2O,1.00,3.00
FAF,0.00,3.00


### Resultado da análise de consistência

A análise das variáveis categóricas não revelou categorias inesperadas ou
inconsistentes. Os valores encontrados correspondem às categorias previstas
para cada atributo, incluindo as sete classes da variável alvo `NObeyesdad`.

Relativamente às variáveis numéricas, os valores mínimos e máximos observados
não evidenciam valores impossíveis ou erros de registo de forma imediata.
Contudo, a existência de valores mais afastados, como a idade máxima de 61
anos ou o peso máximo de 173 kg, não permite concluir que estes sejam erros.

Assim, não foram realizadas alterações nesta etapa. Os possíveis valores
extremos serão analisados separadamente através de um método de deteção de
outliers.

## 2.4 Análise de possíveis outliers

Após a verificação da consistência dos dados, procede-se à identificação de
possíveis valores extremos nas variáveis numéricas.

Para esta análise será utilizado o método do Intervalo Interquartil (IQR).
Este método considera como potenciais outliers os valores inferiores a
Q1 - 1,5 × IQR ou superiores a Q3 + 1,5 × IQR.

A identificação de um valor como potencial outlier não implica a sua remoção
automática. Os resultados serão analisados antes de ser tomada qualquer decisão
sobre o tratamento destes valores.

In [19]:
# 15. função para identificar possíveis outliers através do IQR

def identificar_outliers_iqr(dataframe, coluna):
    q1 = dataframe[coluna].quantile(0.25)
    q3 = dataframe[coluna].quantile(0.75)
    
    iqr = q3 - q1
    
    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr
    
    outliers = dataframe[
        (dataframe[coluna] < limite_inferior) |
        (dataframe[coluna] > limite_superior)
    ]
    
    return q1, q3, limite_inferior, limite_superior, len(outliers)


# 16. aplicação do método IQR

resultados_outliers = []

for coluna in colunas_numericas:
    q1, q3, limite_inf, limite_sup, n_outliers = identificar_outliers_iqr(
        df, coluna
    )
    
    resultados_outliers.append({
        "Variável": coluna,
        "Q1": q1,
        "Q3": q3,
        "Limite inferior": limite_inf,
        "Limite superior": limite_sup,
        "N.º outliers": n_outliers
    })

tabela_outliers = pd.DataFrame(resultados_outliers)

tabela_outliers.round(2)

,Variável,Q1,Q3,Limite inferior,Limite superior,N.º outliers
0,Age,19.97,26.00,10.92,35.05,167
1,Height,1.63,1.77,1.42,1.98,1
2,Weight,66.00,108.02,2.98,171.04,1
3,FCVC,2.00,3.00,0.50,4.50,0
4,NCP,2.70,3.00,2.24,3.45,577
5,CH2O,1.60,2.45,0.32,3.73,0
6,FAF,0.12,1.68,-2.21,4.01,0


In [20]:
# 17. valores extremos identificados em Height e Weight

for coluna in ["Height", "Weight"]:
    q1 = df[coluna].quantile(0.25)
    q3 = df[coluna].quantile(0.75)
    iqr = q3 - q1
    
    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr
    
    outliers = df[
        (df[coluna] < limite_inferior) |
        (df[coluna] > limite_superior)
    ]
    
    print(f"\nOutliers de {coluna}:")
    print(outliers[[coluna, "Age", "Gender", "NObeyesdad"]])


Outliers de Height:
      Height   Age Gender      NObeyesdad
1961    1.98  20.0   Male  Obesity_Type_I

Outliers de Weight:
      Weight   Age Gender        NObeyesdad
1128   173.0  18.0   Male  Obesity_Type_III


### Interpretação e decisão sobre os outliers

A aplicação do método IQR identificou potenciais outliers nas variáveis `Age`,
`Height`, `Weight` e `NCP`. Por outro lado, não foram identificados valores
extremos em `FCVC`, `CH2O` e `FAF` segundo este método.

Em `Age` foram identificados 167 potenciais outliers. No entanto, os valores
observados encontram-se entre 14 e 61 anos, pelo que as idades classificadas
como extremas pelo método IQR continuam a representar valores plausíveis.

Na variável `NCP` foram identificados 577 potenciais outliers. O elevado número
resulta do intervalo relativamente estreito definido pelo IQR (2,24 a 3,45).
Uma vez que esta variável representa o número de refeições principais e os
valores observados se encontram entre 1 e 4, estes casos não foram considerados
erros.

Em `Height` foi identificado apenas um potencial outlier, correspondente a uma
altura de 1,98 m, enquanto em `Weight` foi identificado um registo com 173 kg.
Apesar de serem valores extremos relativamente à distribuição do dataset,
ambos são valores possíveis no contexto das respetivas variáveis.

Desta forma, optou-se por **manter os potenciais outliers**, uma vez que não
foram encontrados indícios suficientes de que correspondam a erros de registo.
A sua remoção apenas com base no critério estatístico do IQR poderia provocar
perda de informação relevante.

## 2.5 Verificação final da qualidade dos dados

Após a análise e tratamento dos problemas de qualidade identificados, é
realizada uma verificação final do conjunto de dados.

Nesta etapa pretende-se confirmar a dimensão atual do dataset, a inexistência
de valores omissos e registos duplicados e os tipos de dados das variáveis,
garantindo que o conjunto de dados se encontra preparado para as etapas
seguintes da análise.

In [21]:
# 18. verificação final da qualidade dos dados

print("VERIFICAÇÃO FINAL")
print("-" * 40)

print(f"Número de registos: {df.shape[0]}")
print(f"Número de atributos: {df.shape[1]}")
print(f"Valores omissos: {df.isna().sum().sum()}")
print(f"Registos duplicados: {df.duplicated().sum()}")

print("\nTipos de dados:")
print(df.dtypes)

VERIFICAÇÃO FINAL
----------------------------------------
Número de registos: 2087
Número de atributos: 13
Valores omissos: 0
Registos duplicados: 0

Tipos de dados:
Gender                             object
Age                               float64
Height                            float64
Weight                            float64
family_history_with_overweight     object
FAVC                               object
FCVC                              float64
NCP                               float64
CAEC                               object
CH2O                              float64
FAF                               float64
MTRANS                             object
NObeyesdad                         object
dtype: object


## Síntese da exploração e preparação dos dados

A análise da qualidade dos dados permitiu identificar e tratar os principais
problemas existentes no conjunto de dados.

Inicialmente foram identificados 45 valores omissos, distribuídos pelas
variáveis `Age`, `CH2O` e `CAEC`. Os valores das variáveis numéricas foram
imputados através da mediana, enquanto os valores omissos de `CAEC` foram
substituídos pela moda.

Foram também identificados 24 registos duplicados, que foram removidos após
a confirmação de que correspondiam a duplicações completas. O dataset passou,
assim, de 2111 para 2087 registos.

A análise das categorias e dos intervalos das variáveis não revelou
inconsistências evidentes. A aplicação do método IQR identificou potenciais
outliers em algumas variáveis, mas estes foram mantidos por representarem
valores plausíveis e não existirem evidências de erros de registo.

Após estas operações, o conjunto de dados apresenta 2087 registos, 13
atributos, nenhum valor omisso e nenhum registo duplicado.

A normalização das variáveis numéricas não foi realizada nesta etapa, ficando
reservada para a preparação dos dados utilizados no K-Means. Desta forma, o
dataset encontra-se preparado para prosseguir para a análise exploratória,
estatística descritiva e visualização dos dados.

In [ ]:
# 19. guardar dataset preparado

#df.to_csv("Dataset_C_Obesity_Preparado.csv", index=False)

#print("Dataset preparado guardado com sucesso.")